##Installations

Run this block to install the required libraries for processing, vectorization, and generation.

In [ ]:
!pip install langchain langchain-classic langchain-huggingface langchain-community pypdf chromadb sentence-transformers deepeval pandas ollama langchain-classic ragas datasets

##Multi-Format Document Ingestion & Chunking
This block loops through a knowledge_base folder, detects whether a file is a PDF, CSV, or JSON, extracts the text appropriately, and chunks it.

In [2]:
import os
import glob
import json
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader, CSVLoader # langchain is restructuring (splitting this package into smaller ones) so this might has a warning
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def load_and_chunk_documents(kb_folder="knowledge_base/*", chunk_size=1000, chunk_overlap=150):
    print(f"Starting chunking process (Size: {chunk_size}, Overlap: {chunk_overlap})...")
    all_files = glob.glob(kb_folder, recursive=True)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    all_chunks = []

    for file_path in all_files:
        if not os.path.isfile(file_path): # skip if the path is anything else but a file (folder)
          continue

        file_name = os.path.basename(file_path)
        try:
            # 1. Handle PDFs (Frameworks, Threat Reports)
            if file_path.endswith('.pdf'):
                loader = PyPDFLoader(file_path)
                docs = loader.load()
                chunks = text_splitter.split_documents(docs)
                all_chunks.extend(chunks)

            # 2. Handle CSVs (e.g., Sigma Rules, tabular data)
            elif file_path.endswith('.csv'):
                loader = CSVLoader(file_path)
                docs = loader.load()
                chunks = text_splitter.split_documents(docs)
                all_chunks.extend(chunks)

            # 3. Handle JSONs (e.g., MITRE ATT&CK matrices)
            elif file_path.endswith('.json'):
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    text_content = json.dumps(data, indent=2)
                    doc = Document(page_content=text_content, metadata={"source": file_name})
                    chunks = text_splitter.split_documents([doc])
                    all_chunks.extend(chunks)

        except Exception as e:
            print(f"Error loading {file_name}: {e}")

    print(f"Total chunks generated: {len(all_chunks)}")
    return all_chunks

# Execute baseline chunking
chunks_baseline = load_and_chunk_documents(chunk_size=1000)

/tmp/ipykernel_4547/646928289.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, CSVLoader


Starting chunking process (Size: 1000, Overlap: 150)...
Total chunks generated: 0


##Creating the Vector Store (ChromaDB)
This block takes the thousands of chunks we just created, converts them into dense embeddings using an open-source model, and stores them locally.

In [8]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Use a fast, open-source embedding model
# warning might appear because the saved model weights include static 'position_ids', but the modern
# transformers library generates them dynamically at runtime, creating a safe mismatch.
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

def build_vector_store(chunks, persist_dir="./chroma_db_baseline"):
    print(f"Building ChromaDB in {persist_dir} with {len(chunks)} chunks...")
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_dir
    )
    print("Vector database built successfully.")
    return vector_store

# Execute baseline database build
vector_store_baseline = build_vector_store(chunks_baseline)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building ChromaDB in ./chroma_db_baseline with 1270 chunks...
Vector database built successfully.


##The RAG System (Generation Module)
Now that the database is built, this block maps the practitioner's query to the retrieval input and uses the LLM to output concise guidelines based strictly on the retrieved chunks.

In [36]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Initialize the local LLM
llm = Ollama(model="llama3")

def build_native_rag_chain(vector_store, top_k=3):
    # Create the retriever with the experimental top_k value
    retriever = vector_store.as_retriever(search_kwargs={"k": top_k})

    # System prompt forcing the LLM to strictly use context and cite sources
    system_prompt = (
        "You are an expert incident response practitioner. "
        "Use the following retrieved context to generate 3 to 5 concise guidelines. "
        "CRITICAL RULE 1: You MUST cite the exact source document name (e.g., [filename.pdf]) for every step. Do not just use numbers like [1]. "
        "CRITICAL RULE 2: Do NOT recommend expensive enterprise tools like SIEMs or assume the user has a massive SOC team. Keep advice universally applicable for low-resource environments. "
        "Context:\n{context}"
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}"),
    ])

    question_answer_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)

    return rag_chain

# Execute baseline RAG setup (Top-K = 3)
native_rag_chain = build_native_rag_chain(vector_store_baseline, top_k=3)

## Local Server Initialization & Automated Evaluation
This block automatically boots the local Ollama server in the background, it then runs the evaluation queries through the Native RAG pipeline and scores the outputs using DeepEval's local judge.

In [37]:
# 1. Install missing dependencies (zstd for extraction, pciutils for GPU)
!sudo apt install -y zstd pciutils > /dev/null

# 2. Install Ollama natively
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start the server completely detached from the Colab cell
!nohup ollama serve > ollama_server.log 2>&1 </dev/null &

# 4. Give the server a few seconds to wake up
!sleep 5

# 5. Download the Llama 3 model
!ollama pull llama3



>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [ ]:
# Block 5: DeepEval Custom Metrics & Automated Evaluation
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import OllamaModel
from deepeval import evaluate

print("Connecting to local Llama 3 server...")

# 1. Define the local judge (Points to the server we started in Cell 1)
local_judge_model = OllamaModel(
    model="llama3",
    base_url="http://localhost:11434",
    temperature=0
)

# 2. Bias & Inclusivity Metric
inclusivity_metric = GEval(
    name="Inclusivity and Resource Bias",
    criteria="Determine if the generated guidelines are universally applicable or if they possess a resource bias (e.g., assuming the user has access to expensive enterprise SIEM tools, massive SOC teams, etc.).",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=local_judge_model,
)

# 3. Transparency Metric
transparency_metric = GEval(
    name="Source Transparency",
    criteria="Determine whether the actual output explicitly cites the source documents provided in the retrieval context.",
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT],
    model=local_judge_model,
)

# 4. Generate Answers and Run Evaluation
eval_queries = [
    "What are the immediate containment steps for a compromised cloud server?",
    "How do we safely capture volatile memory during an active breach?",
    "What is the communication protocol when a massive ransomware event occurs?"
]

deep_eval_test_cases = []
print("Running evaluation queries through Native RAG...")

for query in eval_queries:
    # Generate response from your existing Native RAG chain
    response = native_rag_chain.invoke({"input": query})

    # Package into DeepEval format
    test_case = LLMTestCase(
        input=query,
        actual_output=response["answer"],
        retrieval_context=[doc.page_content for doc in response["context"]]
    )
    deep_eval_test_cases.append(test_case)

print(f"Generated {len(deep_eval_test_cases)} test cases. Starting DeepEval scoring...")

# 5. Execute the evaluation
_ = evaluate(deep_eval_test_cases, metrics=[inclusivity_metric, transparency_metric])

## Extended Evaluation: RAGAS Metrics

This block reformats the test cases we just generated and passes them through a secondary evaluation framework (RAGAS).

In [ ]:
from datasets import Dataset
from ragas import evaluate as ragas_evaluate
from ragas.metrics import answer_relevancy, faithfulness

print("Formatting data for RAGAS...")

# 1. Extract the data you already generated during the DeepEval loop
ragas_data = {
    "question": [tc.input for tc in deep_eval_test_cases],
    "answer": [tc.actual_output for tc in deep_eval_test_cases],
    "contexts": [tc.retrieval_context for tc in deep_eval_test_cases],
}

dataset = Dataset.from_dict(ragas_data)

print("Running RAGAS metrics...")

# 2. Evaluate using the same local LLM and Embeddings you defined in Block 4
ragas_results = ragas_evaluate(
    dataset=dataset,
    metrics=[answer_relevancy, faithfulness],
    llm=llm,                         # Your Ollama LangChain object
    embeddings=embedding_model,      # Your HuggingFaceEmbeddings object
)

# 3. Display Results
df_ragas = ragas_results.to_pandas()
display(df_ragas)